### Tools

 Tools in LangChain allow an LLM to interact with external systems to perform tasks such as web search, database queries, API calls, or code execution. A tool consists of a schema (name, description, and parameters) and a function that executes the task.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:llama-3.3-70b-versatile")
response=model.invoke("why do parrot talk")
response.content

In [ ]:
from langchain.tools import tool
@tool
def get_weather(location:str)->str:
    """ Get the weather at a location """
    return f"It is sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [ ]:
response=model_with_tools.invoke("What's the weather in New York")
print(response)

for tool_call in response.tool_calls:
    # view tool calls made by model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

### Tool Excution Loops

In [ ]:
# Step 1: User message
messages = [
    {
        "role": "user",
        "content": "What's the weather in New Delhi?"
    }
]

# Model decides whether it needs a tool
ai_msg = model_with_tools.invoke(messages)

# Add AI message to conversation history
messages.append(ai_msg)

# Step 2: Execute tool(s)
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Give tool result back to model
final_response = model_with_tools.invoke(messages)

print(final_response.text)